In [ ]:
from sqlalchemy.util.preloaded import orm_util

from external.lammps.examples.ELASTIC_T.BORN_MATRIX.Silicon.elastic import \
  origin
%env JAX_PLATFORMS=cpu

In [ ]:
import numpy as onp
import matplotlib.pyplot as plt

import mdtraj

from pathlib import Path

# Physical constants.
kB = 0.001987 # Boltzmann constant in kcal/mol/K.
t0 = 48.8882129 # Time unit in fs.
hbar = 15.18 / t0 # Reduced Planck constant from kcal s/mol to consistent units.

print(hbar)

In [ ]:
def integrate(dir):
  _, _, T, _, P = dir.name.split('_')
  T, P = int(T), int (P)

  V = onp.loadtxt(dir / "bcc_avgvol.txt")[-1, -1]
  k = onp.loadtxt(dir / "bcc_force_constant.txt")
  Vpa = onp.loadtxt(dir / "bcc_vol_per_atom.txt")

  natoms = int(onp.round(V / Vpa))

  # Input parameters.
  m = 47.867 # Titanium mass in amu.

  ################################################################################
  # Lambda integration [Eq.(12) in the paper].
  ################################################################################


  tE, tlambd = [], []
  # Forward integration.
  dE, lamb = onp.loadtxt(dir / ('forward_%dK.dat' % T), unpack=True)
  I_forw = onp.trapezoid(dE,lamb)
  tE.append(dE)
  tlambd.append(lamb)

  # Backward integration.
  dE, lamb = onp.loadtxt(dir / ('backward_%dK.dat' % T), unpack=True)
  I_back = onp.trapezoid(dE,lamb)
  # Compute reversible work.
  tE.append(dE)
  tlambd.append(lamb)

  W = (I_forw-I_back) / 2

  ################################################################################
  # Compute free energy.
  ################################################################################

  # Define harmonic reference system free energy [Eq.(15) in the paper].
  omega = onp.sqrt(k/m)

  F_harm = 3*natoms*kB*T * onp.log(hbar*omega/(kB*T)) # energy units

  # Fixed center of mass correction [Eq.(24) in the paper].
  F_CM = (kB*T)*onp.log((natoms/V) * (2*onp.pi*kB*T / (natoms*k))**(3/2)) # [eV].

  # Compute absolute free energy per atom [Eq.(16) in the paper] and save data.
  F = (F_harm + W + F_CM) / natoms # [eV/atom].

  print(f"Found run at T={T}K, P={P}GPa, V={V} A^3, k={k} kcal/mol/A^2, N={natoms} atoms and F={F} kcal/mol/atom.")

  return T, P, V, F, onp.concatenate(tE, axis=0), onp.concatenate(tlambd, axis=0)

  ################################################################################

In [ ]:
ref_dir = Path("../lammps/solidfe/output/titanium__MACE_r_cutoff_0.5_2025_5_4_21abc4af-aea1-4549-a9a5-981f6324c2e5/202508212118")
print([int(run.name.split('_')[4]) for run in ref_dir.glob("bcc_*")])
sort_idx = onp.argsort([int(run.name.split('_')[4]) for run in ref_dir.glob("bcc_*")])

runs = onp.fromiter(ref_dir.glob("bcc_*"), dtype=Path)[sort_idx]

T = []
P = []
V = []
F = []
E = []
lmbd = []

for run in runs:
    Ti, Pi, Vi, Fi, Ei, lmbdi = integrate(run)
    T.append(Ti)
    P.append(Pi)
    V.append(Vi)
    F.append(Fi)
    E.append(Ei)
    lmbd.append(lmbdi)

sort_idx = onp.argsort(P)
T = onp.array(T)[sort_idx]
P = onp.array(P)[sort_idx]
V = onp.array(V)[sort_idx]
F = onp.array(F)[sort_idx]
E = onp.array(E)[sort_idx]
lmbd = onp.array(lmbd)[sort_idx]


In [ ]:
plt.plot(P, V)

In [ ]:
plt.plot(P, F)

In [ ]:
plt.semilogy(lmbd[:, :(E.shape[1] // 2)].T, (E[:, :(E.shape[1] // 2)] - E[:, (E.shape[1] // 2):]).T)
plt.ylim([-1e5 , 1e8 ])

In [ ]:
E.shape[1] // 2

In [ ]:
E[:, :(E.shape[1] // 2)].shape